In [ ]:
from __future__ import annotations

import pickle

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from scipy.linalg import subspace_angles
from scipy.stats import ttest_ind

import manifold_dynamics.neural_utils as nu
import manifold_dynamics.paths as pth
import manifold_dynamics.tuning_utils as tut
import visionlab_utils.storage as vst

In [ ]:
# ROI name, should be 3 parts (index.label.category)
target = "08.MF1.F"
target_parts = target.split(".")
roi_label = f"{int(target_parts[0]):02d}.{target_parts[1]}.{target_parts[2]}"

# load in multi-session ROI data, binned to PSTH
raster_4d = nu.significant_trial_raster(target, alpha=0.05, bin_size_ms=20)

# get top-k value for ROI
topk_local = vst.fetch(f"{pth.OTHERS}/topk_vals.pkl")
with open(topk_local, "rb") as f:
    topk_vals = pickle.load(f)
top_k = int(topk_vals[roi_label]["k"])

print(f"Resolved ROI target: {target}")
print(f"Using top-k = {top_k}")
print(f"Raster shape after binning {raster_4d.shape}")

# shape (units, time, images)
raster_3d = np.nanmean(raster_4d, axis=3)
scores = tut.rank_images_by_response(raster_3d)

idx_topk = scores[:top_k]
trial_3d = raster_3d[:, :, idx_topk]
print(f"Trial averaged, all data {raster_3d.shape}")
print(f"Trial averaged, top-k shape {trial_3d.shape}")

In [ ]:
# parameters
seed = 0
n_random = 100
n_components = 3
t_start = 100
t_stop = 270

rng = np.random.default_rng(seed)
candidate_idxs = scores[top_k:]

# average over time window
resp = np.nanmean(raster_3d[:, t_start:t_stop, :], axis=1)  # (units, images)
R = resp.T                                                   # (images, units)

# subspaces for top-k and all images
A_top = PCA(n_components=n_components).fit(R[idx_topk]).components_.T
A_all = PCA(n_components=n_components).fit(R).components_.T

# direct comparisons
angles_top_all_deg = np.degrees(subspace_angles(A_top, A_all))
print(f"Top-k vs all-images angles: {angles_top_all_deg}")

# pre-sample random image sets
random_idxs = np.stack(
    [rng.choice(candidate_idxs, size=top_k, replace=False) for _ in range(n_random)],
    axis=0,
)

# top-vs-random
angles_top_rand_deg = np.zeros((n_random, n_components))
for i, idx_rand in enumerate(random_idxs):
    A_rand = PCA(n_components=n_components).fit(R[idx_rand]).components_.T
    angles_top_rand_deg[i] = np.degrees(subspace_angles(A_top, A_rand))

print(f"Top-vs-random mean angles: {angles_top_rand_deg.mean(axis=0)}")

# random-vs-random
angles_rand_rand_deg = np.zeros((n_random, n_components))
for i, idx_a in enumerate(random_idxs):
    idx_b = rng.choice(candidate_idxs, size=top_k, replace=False)
    A_rand_a = PCA(n_components=n_components).fit(R[idx_a]).components_.T
    A_rand_b = PCA(n_components=n_components).fit(R[idx_b]).components_.T
    angles_rand_rand_deg[i] = np.degrees(subspace_angles(A_rand_a, A_rand_b))

print(f"Random-vs-random mean angles: {angles_rand_rand_deg.mean(axis=0)}")

# all-images-vs-random
angles_all_rand_deg = np.zeros((n_random, n_components))
for i, idx_rand in enumerate(random_idxs):
    A_rand = PCA(n_components=n_components).fit(R[idx_rand]).components_.T
    angles_all_rand_deg[i] = np.degrees(subspace_angles(A_all, A_rand))

print(f"All-images-vs-random mean angles: {angles_all_rand_deg.mean(axis=0)}")

# overall scalar summaries
top_rand_mean = angles_top_rand_deg.mean(axis=1)
rand_rand_mean = angles_rand_rand_deg.mean(axis=1)
all_rand_mean = angles_all_rand_deg.mean(axis=1)
top_all_mean = angles_top_all_deg.mean()

# Top k is further from a random set than would be expected by change
print(f"\nTop-vs-random mean: {top_rand_mean.mean()}")
print(f"Random mean: {rand_rand_mean.mean()}")

# these 2 are similar
# my interpretation: top-k subspace is not further than All compared to Random
# or: global contains random + top-k
print(f"All-vs-random mean: {all_rand_mean.mean()}")
print(f"Top-vs-all mean: {top_all_mean}\n")

In [ ]:
df = pd.DataFrame({
    "vals": np.concatenate([
        top_rand_mean,
        rand_rand_mean,
        all_rand_mean,
        np.repeat(top_all_mean, len(top_rand_mean)),  # scalar → match length
    ]),
    "type": (
        ["top-rand"] * len(top_rand_mean) +
        ["rand-rand"] * len(rand_rand_mean) +
        ["all-rand"] * len(all_rand_mean) +
        ["top-all"] * len(top_rand_mean)
    )
})

fig,ax = plt.subplots(1,1)
sns.boxplot(data=df, x="type", y="vals", ax=ax)

ax.set_ylabel("Mean principal angle (deg)")
ax.set_xlabel("")

In [ ]:
'''
now over time...

3/4 show this weird block pattern for top and not all, around ~120 msec

that last 1/4 shows an interesting all pattern...

maybe something?
'''
window_size = 100
step = 10
time_centers = np.arange(0, 450 - window_size, step)

subspaces = []
for t in time_centers:
    R_t = np.nanmean(raster_3d[:, t:t+window_size, :], axis=1).T
    R_top_t = R_t[idx_topk]
    A_t = PCA(n_components=2).fit(R_top_t).components_.T
    subspaces.append(A_t)

n_t = len(subspaces)
angles_tt = np.zeros((n_t, n_t))
for i in range(n_t):
    for j in range(n_t):
        ang = subspace_angles(subspaces[i], subspaces[j])
        angles_tt[i, j] = np.degrees(ang).mean()

sns.heatmap(angles_tt, square=True)
plt.title("Top")
plt.show()

subspaces = []
for t in time_centers:
    R_t = np.nanmean(raster_3d[:, t:t+window_size, :], axis=1).T
    R_top_t = R_t[random_idxs[10]]
    A_t = PCA(n_components=2).fit(R_top_t).components_.T
    subspaces.append(A_t)

n_t = len(subspaces)
angles_tt = np.zeros((n_t, n_t))
for i in range(n_t):
    for j in range(n_t):
        ang = subspace_angles(subspaces[i], subspaces[j])
        angles_tt[i, j] = np.degrees(ang).mean()

sns.heatmap(angles_tt, square=True)
plt.title("1 random")
plt.show()

subspaces = []
for t in time_centers:
    R_t = np.nanmean(raster_3d[:, t:t+window_size, :], axis=1).T
    R_top_t = R_t
    A_t = PCA(n_components=2).fit(R_top_t).components_.T
    subspaces.append(A_t)

n_t = len(subspaces)
angles_tt = np.zeros((n_t, n_t))
for i in range(n_t):
    for j in range(n_t):
        ang = subspace_angles(subspaces[i], subspaces[j])
        angles_tt[i, j] = np.degrees(ang).mean()

sns.heatmap(angles_tt, square=True)
plt.title("All")
plt.show()

In [ ]:
subspaces = []
for t in time_centers:
    R_t = np.nanmean(raster_3d[:, t:t+window_size, :], axis=1).T
    R_top_t = R_t[random_idxs[0]]
    A_t = PCA(n_components=2).fit(R_top_t).components_.T
    subspaces.append(A_t)

n_t = len(subspaces)
angles_tt = np.zeros((n_t, n_t))
for i in range(n_t):
    for j in range(n_t):
        ang = subspace_angles(subspaces[i], subspaces[j])
        angles_tt[i, j] = np.degrees(ang).mean()

sns.heatmap(angles_tt, square=True)
plt.title("1 random")
plt.show()